# Heart Rate Variability and Autonomic Function

**Target Audience:** Cardiology residents, critical care fellows, researchers

**Learning Objectives:**
1. Understand HRV as a window into autonomic function
2. Compute time-domain and frequency-domain HRV metrics
3. Interpret HRV in clinical contexts (MI, heart failure, sepsis)
4. Explore heart-brain coupling through computational modeling

**Clinical Relevance:**
- Reduced HRV → Increased mortality post-MI
- Sympathovagal imbalance in heart failure
- Autonomic dysfunction in critical illness

---

In [ ]:
# Imports
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from src.cardiac import VanDerPolOscillator
from src.neural import FitzHughNagumo
from src.coupling import HeartBrainCouplingModel, CouplingParameters
from src.validation.metrics import (
    compute_hrv_metrics,
    extract_rr_intervals_from_trajectory,
    classify_hrv_status,
)
from src.validation.benchmarks import PhysiologicalBenchmarks

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ All imports successful")
print("\nReady to explore heart rate variability!")

## Part 1: Simulating Heart-Brain Coupling

### Physiological Background

The heart and brain communicate bidirectionally:

**Brain → Heart (Efferent Pathway)**
- Vagal nerve (parasympathetic): Fast, inhibitory, 50-150ms delay
- Sympathetic nerves: Slow, excitatory, 150-300ms delay

**Heart → Brain (Afferent Pathway)**
- Baroreceptors detect pressure changes
- Signals travel to brainstem (100-200ms delay)
- Modulate autonomic output

### Our Model
- **Neural oscillator (FitzHugh-Nagumo):** Represents autonomic neural activity
- **Cardiac oscillator (Van der Pol):** Represents intrinsic cardiac rhythm
- **Coupling:** Time-delayed bidirectional influence

This creates **heart rate variability** through neural-cardiac interaction.

In [ ]:
# Create baseline healthy model
def create_healthy_model():
    """Create heart-brain model with healthy parameters."""
    neural = FitzHughNagumo(
        a=0.7,
        b=0.8,
        c=3.0,
        stimulus_amplitude=0.3  # Tonic neural drive
    )
    
    cardiac = VanDerPolOscillator(
        mu=1.5,  # Moderate nonlinearity
        omega=1.0,  # Natural frequency ~60 bpm
        damping=0.1
    )
    
    coupling = CouplingParameters(
        neural_to_cardiac_gain=0.5,  # Strong vagal influence
        cardiac_to_neural_gain=0.3,  # Moderate baroreceptor feedback
        neural_to_cardiac_delay=0.10,  # Vagal delay (100ms)
        cardiac_to_neural_delay=0.15   # Baroreceptor delay (150ms)
    )
    
    return HeartBrainCouplingModel(neural, cardiac, coupling)

# Simulate healthy state
print("Simulating healthy heart-brain coupling...")
healthy_model = create_healthy_model()

initial_state = (0.0, 0.0, 1.0, 0.0)  # (v, w, x, y)
t_span = (0.0, 120.0)  # 2 minutes
dt = 0.001  # 1ms timestep

trajectory_healthy = healthy_model.simulate(initial_state, t_span, dt)

# Extract time series
times, neural_states, cardiac_states = healthy_model.extract_series(trajectory_healthy)

# Extract RR intervals
rr_intervals_healthy = extract_rr_intervals_from_trajectory(
    trajectory_healthy,
    cardiac_component_index=2,
    threshold=0.0
)

print(f"✓ Simulation complete")
print(f"  Duration: {times[-1]:.1f} seconds")
print(f"  Total heartbeats: {len(rr_intervals_healthy)}")
print(f"  Mean RR interval: {np.mean(rr_intervals_healthy):.1f} ms")
print(f"  Mean heart rate: {60000 / np.mean(rr_intervals_healthy):.1f} bpm")

### Visualize Coupled Dynamics

In [ ]:
# Plot first 10 seconds to see detail
plot_duration = 10.0
plot_indices = times <= plot_duration

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Neural activity
ax1 = axes[0]
v_values = [v for v, w in neural_states]
ax1.plot(times[plot_indices], np.array(v_values)[plot_indices], 'b-', linewidth=1.5)
ax1.set_ylabel('Neural Activity (v)', fontsize=12, fontweight='bold')
ax1.set_title('Heart-Brain Coupling: Coupled Oscillator Dynamics', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, plot_duration)

# Cardiac activity
ax2 = axes[1]
x_values = [x for x, y in cardiac_states]
ax2.plot(times[plot_indices], np.array(x_values)[plot_indices], 'r-', linewidth=1.5)
ax2.set_ylabel('Cardiac Activity (x)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, plot_duration)

# RR intervals (tachogram)
ax3 = axes[2]
# Find peak times
peak_times = []
for i in range(1, len(x_values)-1):
    if x_values[i] > x_values[i-1] and x_values[i] > x_values[i+1] and x_values[i] > 0:
        peak_times.append(times[i])

rr_intervals_plot = [(peak_times[i+1] - peak_times[i]) * 1000 for i in range(len(peak_times)-1)]
rr_times = peak_times[1:len(rr_intervals_plot)+1]

# Plot only first 10 seconds
plot_indices_rr = [i for i, t in enumerate(rr_times) if t <= plot_duration]
if plot_indices_rr:
    ax3.plot([rr_times[i] for i in plot_indices_rr], 
            [rr_intervals_plot[i] for i in plot_indices_rr], 
            'go-', linewidth=1.5, markersize=6)
    ax3.set_ylabel('RR Interval (ms)', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(0, plot_duration)

    # Add mean line
    mean_rr = np.mean([rr_intervals_plot[i] for i in plot_indices_rr])
    ax3.axhline(y=mean_rr, color='k', linestyle='--', alpha=0.5, label=f'Mean = {mean_rr:.1f} ms')
    ax3.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  • Neural and cardiac oscillators are COUPLED")
print("  • RR intervals show variability (not perfectly constant)")
print("  • This variability reflects autonomic modulation")

## Part 2: HRV Metrics - Time and Frequency Domain

### HRV Metrics from Task Force (1996)

**Time Domain:**
- **SDNN:** Standard deviation of NN intervals (overall HRV)
- **RMSSD:** Root mean square of successive differences (short-term/parasympathetic)
- **pNN50:** % of intervals differing by >50ms (parasympathetic)

**Frequency Domain:**
- **VLF (0.003-0.04 Hz):** Very low frequency (hormonal, thermoregulation)
- **LF (0.04-0.15 Hz):** Low frequency (sympathetic + parasympathetic)
- **HF (0.15-0.4 Hz):** High frequency (parasympathetic, respiratory sinus arrhythmia)
- **LF/HF ratio:** Sympathovagal balance

In [ ]:
# Compute HRV metrics for healthy state
hrv_healthy = compute_hrv_metrics(rr_intervals_healthy)
status_healthy = classify_hrv_status(hrv_healthy)

# Get benchmarks
benchmarks = PhysiologicalBenchmarks()

# Display results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time domain metrics
ax1 = axes[0, 0]
time_metrics = ['sdnn_ms', 'rmssd_ms', 'pnn50_pct']
time_values = [hrv_healthy[m] for m in time_metrics]
time_labels = ['SDNN', 'RMSSD', 'pNN50']

bars = ax1.bar(time_labels, time_values, color=['steelblue', 'coral', 'lightgreen'], edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Value', fontsize=12, fontweight='bold')
ax1.set_title('Time-Domain HRV Metrics', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar, val in zip(bars, time_values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add reference lines
ax1.axhline(y=benchmarks.hrv.sdnn.min_value, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax1.text(0.5, benchmarks.hrv.sdnn.min_value + 5, 'SDNN threshold', fontsize=9)

# Frequency domain metrics
ax2 = axes[0, 1]
freq_metrics = ['vlf_power_ms2', 'lf_power_ms2', 'hf_power_ms2']
freq_values = [hrv_healthy[m] for m in freq_metrics]
freq_labels = ['VLF\n(0.003-0.04 Hz)', 'LF\n(0.04-0.15 Hz)', 'HF\n(0.15-0.4 Hz)']
colors_freq = ['purple', 'orange', 'green']

bars = ax2.bar(freq_labels, freq_values, color=colors_freq, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Power (ms²)', fontsize=12, fontweight='bold')
ax2.set_title('Frequency-Domain HRV Metrics', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, freq_values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# LF/HF ratio (sympathovagal balance)
ax3 = axes[1, 0]
lf_hf = hrv_healthy['lf_hf_ratio']
ax3.barh(['LF/HF Ratio'], [lf_hf], color='teal', edgecolor='black', linewidth=2)
ax3.set_xlabel('Ratio', fontsize=12, fontweight='bold')
ax3.set_title('Sympathovagal Balance', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')
ax3.set_xlim(0, 3)

# Add reference zones
ax3.axvspan(0.5, 2.5, alpha=0.2, color='green', label='Normal range')
ax3.axvspan(2.5, 3.0, alpha=0.2, color='red', label='Sympathetic dominance')
ax3.axvspan(0, 0.5, alpha=0.2, color='blue', label='Parasympathetic dominance')
ax3.text(lf_hf + 0.1, 0, f'{lf_hf:.2f}', va='center', fontsize=12, fontweight='bold')
ax3.legend(loc='lower right', fontsize=9)

# Clinical interpretation
ax4 = axes[1, 1]
ax4.axis('off')

interpretation = f"""
HRV ANALYSIS REPORT
{'='*50}

TIME DOMAIN:
  SDNN:    {hrv_healthy['sdnn_ms']:.1f} ms  (Normal: >100 ms)
  RMSSD:   {hrv_healthy['rmssd_ms']:.1f} ms  (Normal: 20-100 ms)
  pNN50:   {hrv_healthy['pnn50_pct']:.1f}%

FREQUENCY DOMAIN:
  VLF:     {hrv_healthy['vlf_power_ms2']:.1f} ms²
  LF:      {hrv_healthy['lf_power_ms2']:.1f} ms²
  HF:      {hrv_healthy['hf_power_ms2']:.1f} ms²
  LF/HF:   {hrv_healthy['lf_hf_ratio']:.2f}  (Normal: 0.5-2.5)

AUTONOMIC STATUS:
  Classification: {status_healthy.replace('_', ' ').title()}

CLINICAL INTERPRETATION:
  ✓ HRV is preserved
  ✓ Sympathovagal balance maintained
  ✓ Normal autonomic function
"""

ax4.text(0.1, 0.5, interpretation, fontsize=10, family='monospace',
        verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n✓ HRV analysis complete for healthy state")

## Part 3: Pathological States - Reduced HRV

### Clinical Scenarios

**Post-Myocardial Infarction:**
- Reduced HRV associated with increased mortality (Kleiger et al., 1987)
- Impaired baroreflex sensitivity
- Sympathetic hyperactivity

**Heart Failure:**
- Markedly reduced HRV
- High LF/HF ratio (sympathetic dominance)
- Poor prognostic marker

**Sepsis:**
- Loss of HRV complexity
- Uncoupling of autonomic regulation

In [ ]:
# Simulate pathological states

# 1. Post-MI (reduced vagal tone)
def create_post_mi_model():
    """Reduced parasympathetic influence."""
    neural = FitzHughNagumo(a=0.7, b=0.8, c=3.0, stimulus_amplitude=0.1)  # Reduced
    cardiac = VanDerPolOscillator(mu=1.5, omega=1.0, damping=0.1)
    coupling = CouplingParameters(
        neural_to_cardiac_gain=0.2,  # Reduced vagal influence
        cardiac_to_neural_gain=0.2,  # Reduced baroreflex
        neural_to_cardiac_delay=0.10,
        cardiac_to_neural_delay=0.15
    )
    return HeartBrainCouplingModel(neural, cardiac, coupling)

# 2. Heart failure (sympathetic dominance)
def create_hf_model():
    """High sympathetic, low parasympathetic."""
    neural = FitzHughNagumo(a=0.7, b=0.8, c=3.0, stimulus_amplitude=0.5)  # High sympathetic
    cardiac = VanDerPolOscillator(mu=1.5, omega=1.1, damping=0.05)  # Higher rate
    coupling = CouplingParameters(
        neural_to_cardiac_gain=0.8,  # Strong sympathetic
        cardiac_to_neural_gain=0.1,  # Weak baroreflex
        neural_to_cardiac_delay=0.20,  # Sympathetic delay
        cardiac_to_neural_delay=0.15
    )
    return HeartBrainCouplingModel(neural, cardiac, coupling)

# Simulate
print("Simulating pathological states...")
print("  1. Post-MI (reduced vagal tone)...")
model_mi = create_post_mi_model()
traj_mi = model_mi.simulate(initial_state, (0, 120), dt)
rr_mi = extract_rr_intervals_from_trajectory(traj_mi, cardiac_component_index=2)
hrv_mi = compute_hrv_metrics(rr_mi)
status_mi = classify_hrv_status(hrv_mi)

print("  2. Heart failure (sympathetic dominance)...")
model_hf = create_hf_model()
traj_hf = model_hf.simulate(initial_state, (0, 120), dt)
rr_hf = extract_rr_intervals_from_trajectory(traj_hf, cardiac_component_index=2)
hrv_hf = compute_hrv_metrics(rr_hf)
status_hf = classify_hrv_status(hrv_hf)

print("✓ Simulations complete\n")

# Compare metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# SDNN comparison
ax1 = axes[0, 0]
states = ['Healthy', 'Post-MI', 'Heart Failure']
sdnn_values = [hrv_healthy['sdnn_ms'], hrv_mi['sdnn_ms'], hrv_hf['sdnn_ms']]
colors = ['green', 'orange', 'red']

bars = ax1.bar(states, sdnn_values, color=colors, edgecolor='black', linewidth=2)
ax1.set_ylabel('SDNN (ms)', fontsize=12, fontweight='bold')
ax1.set_title('Standard Deviation of NN Intervals', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Normal threshold')
ax1.axhline(y=50, color='darkred', linestyle='--', linewidth=2, label='High risk threshold')
ax1.legend(fontsize=9)

for bar, val in zip(bars, sdnn_values):
    ax1.text(bar.get_x() + bar.get_width()/2., val,
            f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# RMSSD comparison
ax2 = axes[0, 1]
rmssd_values = [hrv_healthy['rmssd_ms'], hrv_mi['rmssd_ms'], hrv_hf['rmssd_ms']]
bars = ax2.bar(states, rmssd_values, color=colors, edgecolor='black', linewidth=2)
ax2.set_ylabel('RMSSD (ms)', fontsize=12, fontweight='bold')
ax2.set_title('Root Mean Square of Successive Differences\n(Parasympathetic Activity)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, rmssd_values):
    ax2.text(bar.get_x() + bar.get_width()/2., val,
            f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# LF/HF ratio comparison
ax3 = axes[1, 0]
lf_hf_values = [hrv_healthy['lf_hf_ratio'], hrv_mi['lf_hf_ratio'], hrv_hf['lf_hf_ratio']]
bars = ax3.bar(states, lf_hf_values, color=colors, edgecolor='black', linewidth=2)
ax3.set_ylabel('LF/HF Ratio', fontsize=12, fontweight='bold')
ax3.set_title('Sympathovagal Balance', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')
ax3.axhspan(0.5, 2.5, alpha=0.2, color='green', label='Normal range')
ax3.axhline(y=2.5, color='red', linestyle='--', linewidth=1.5, label='Sympathetic dominance')
ax3.legend(fontsize=9)

for bar, val in zip(bars, lf_hf_values):
    ax3.text(bar.get_x() + bar.get_width()/2., val,
            f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Summary table
ax4 = axes[1, 1]
ax4.axis('off')

comparison_text = f"""
COMPARATIVE HRV ANALYSIS
{'='*55}

               HEALTHY    POST-MI    HEART FAILURE
{'-'*55}
SDNN (ms)      {sdnn_values[0]:6.1f}     {sdnn_values[1]:6.1f}     {sdnn_values[2]:6.1f}
RMSSD (ms)     {rmssd_values[0]:6.1f}     {rmssd_values[1]:6.1f}     {rmssd_values[2]:6.1f}
LF/HF          {lf_hf_values[0]:6.2f}     {lf_hf_values[1]:6.2f}     {lf_hf_values[2]:6.2f}
{'-'*55}

CLINICAL IMPLICATIONS:

Post-MI:
  • SDNN < 100 ms → High mortality risk
  • Reduced RMSSD → Impaired vagal tone
  • Consider: Beta-blockers, ACE-I

Heart Failure:
  • Very low SDNN → Advanced dysfunction
  • High LF/HF → Sympathetic overdrive
  • Consider: Cardiac resynchronization
"""

ax4.text(0.05, 0.5, comparison_text, fontsize=10, family='monospace',
        verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n💡 Key Clinical Insights:")
print("  • Healthy: High HRV, balanced autonomics")
print("  • Post-MI: Reduced HRV, impaired vagal tone")
print("  • Heart Failure: Very low HRV, sympathetic dominance")
print("  • HRV is a powerful prognostic marker!")

## Part 4: Poincaré Plots - Visual HRV Assessment

**Poincaré Plot:** RR(n+1) vs RR(n)

**Interpretation:**
- **SD1:** Short-term variability (perpendicular to line of identity)
- **SD2:** Long-term variability (along line of identity)
- **SD1/SD2 ratio:** Shape of cloud

**Clinical patterns:**
- Healthy: Comet/cigar shape
- Reduced HRV: Tight cluster
- Atrial fibrillation: Scattered cloud

In [ ]:
# Create Poincaré plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rr_datasets = [
    (rr_intervals_healthy, 'Healthy', 'green'),
    (rr_mi, 'Post-MI', 'orange'),
    (rr_hf, 'Heart Failure', 'red')
]

for ax, (rr_data, title, color) in zip(axes, rr_datasets):
    # Create Poincaré plot
    rr_n = rr_data[:-1]
    rr_n1 = rr_data[1:]
    
    ax.scatter(rr_n, rr_n1, alpha=0.6, s=30, color=color, edgecolors='black', linewidth=0.5)
    
    # Add line of identity
    min_rr = min(min(rr_n), min(rr_n1))
    max_rr = max(max(rr_n), max(rr_n1))
    ax.plot([min_rr, max_rr], [min_rr, max_rr], 'k--', linewidth=1.5, alpha=0.5, label='Identity line')
    
    # Calculate SD1 and SD2
    # SD1: standard deviation perpendicular to identity line
    # SD2: standard deviation along identity line
    diff = np.array(rr_n1) - np.array(rr_n)
    sd1 = np.std(diff) / np.sqrt(2)
    
    summ = np.array(rr_n1) + np.array(rr_n)
    sd2 = np.std(summ) / np.sqrt(2)
    
    ax.set_xlabel('RR(n) [ms]', fontsize=11, fontweight='bold')
    ax.set_ylabel('RR(n+1) [ms]', fontsize=11, fontweight='bold')
    ax.set_title(f'{title}\nSD1={sd1:.1f} ms, SD2={sd2:.1f} ms', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Poincaré Plot Interpretation:")
print("  • Healthy: Dispersed cloud (high variability)")
print("  • Post-MI: Tighter cluster (reduced variability)")
print("  • Heart Failure: Very tight cluster (minimal variability)")

## Summary and Clinical Pearls

### 🎯 Key Takeaways

1. **HRV reflects autonomic function**
   - High HRV = Healthy, balanced autonomics
   - Low HRV = Autonomic dysfunction, poor prognosis

2. **Time-domain metrics**
   - SDNN < 50 ms: Severely reduced (high mortality)
   - SDNN 50-100 ms: Moderately reduced
   - SDNN > 100 ms: Normal

3. **Frequency-domain metrics**
   - LF/HF < 0.5: Parasympathetic dominance
   - LF/HF 0.5-2.5: Balanced
   - LF/HF > 2.5: Sympathetic dominance

4. **Clinical applications**
   - Risk stratification post-MI
   - Heart failure prognosis
   - Diabetes autonomic neuropathy
   - Sepsis monitoring

### 📚 Further Reading
- Task Force (1996): HRV standards
- Kleiger et al. (1987): HRV and mortality post-MI
- La Rovere et al. (1998): Baroreflex sensitivity

### 🔬 Next Steps
- Notebook 03: Swan-Ganz catheter waveforms
- Notebook 04: Valsalva maneuver simulation
- Try modifying coupling parameters to see effects

---

© 2025 Multi-Heart-Model Project | MIT License